In [9]:
year       = 2024       
months     = list(range(1, 13))   
overwrite  = False       

StatementMeta(, a0441e3c-c954-4571-8460-6d374d5dac66, 11, Finished, Available, Finished, False)

Imports

In [10]:
import requests
import os
from datetime import datetime

BRONZE_FILES = "/lakehouse/default/Files/taxi"
os.makedirs(BRONZE_FILES, exist_ok=True)

BASE_URL = (
    "https://d37ci6vzurychx.cloudfront.net/trip-data/"
    "yellow_tripdata_{year}-{month:02d}.parquet"
)

StatementMeta(, a0441e3c-c954-4571-8460-6d374d5dac66, 12, Finished, Available, Finished, False)

Helper function

In [11]:
def download_parquet(year: int, month: int, dest_dir: str, overwrite: bool) -> dict:
    url      = BASE_URL.format(year=year, month=month)
    filename = f"yellow_tripdata_{year}-{month:02d}.parquet"
    dest     = os.path.join(dest_dir, filename)

    if os.path.exists(dest) and not overwrite:
        print(f"  [SKIP]  {filename} already exists.")
        return {"file": filename, "status": "skipped", "bytes": os.path.getsize(dest)}

    print(f"  [GET]   {url}")
    try:
        with requests.get(url, stream=True, timeout=300) as r:
            r.raise_for_status()
            total = 0
            with open(dest, "wb") as f:
                for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):  # 8 MB chunks
                    f.write(chunk)
                    total += len(chunk)
        size_mb = total / 1_048_576
        print(f"  [OK]    {filename}  ({size_mb:.1f} MB)")
        return {"file": filename, "status": "downloaded", "bytes": total}
    except requests.HTTPError as e:
        print(f"  [WARN]  {filename} – HTTP {e.response.status_code}. File may not exist yet.")
        return {"file": filename, "status": "http_error", "code": e.response.status_code}
    except Exception as e:
        print(f"  [ERROR] {filename} – {e}")
        return {"file": filename, "status": "error", "error": str(e)}

StatementMeta(, a0441e3c-c954-4571-8460-6d374d5dac66, 13, Finished, Available, Finished, False)

In [12]:
print(f"  NYC Taxi Bronze Ingestion – Year {year}")
print(f"  Target: {BRONZE_FILES}")
print(f"  Started: {datetime.utcnow().isoformat()}Z")

results = []
for m in months:
    result = download_parquet(year, m, BRONZE_FILES, overwrite)
    results.append(result)

StatementMeta(, a0441e3c-c954-4571-8460-6d374d5dac66, 14, Finished, Available, Finished, False)

  NYC Taxi Bronze Ingestion – Year 2024
  Target: /lakehouse/default/Files/taxi
  Started: 2026-05-21T19:34:47.052472Z
  [SKIP]  yellow_tripdata_2024-01.parquet already exists.
  [SKIP]  yellow_tripdata_2024-02.parquet already exists.
  [SKIP]  yellow_tripdata_2024-03.parquet already exists.
  [SKIP]  yellow_tripdata_2024-04.parquet already exists.
  [SKIP]  yellow_tripdata_2024-05.parquet already exists.
  [SKIP]  yellow_tripdata_2024-06.parquet already exists.
  [SKIP]  yellow_tripdata_2024-07.parquet already exists.
  [SKIP]  yellow_tripdata_2024-08.parquet already exists.
  [SKIP]  yellow_tripdata_2024-09.parquet already exists.
  [SKIP]  yellow_tripdata_2024-10.parquet already exists.
  [SKIP]  yellow_tripdata_2024-11.parquet already exists.
  [SKIP]  yellow_tripdata_2024-12.parquet already exists.


Check

In [13]:
downloaded = [r for r in results if r["status"] == "downloaded"]
skipped    = [r for r in results if r["status"] == "skipped"]
errors     = [r for r in results if r["status"] not in ("downloaded", "skipped")]

total_mb = sum(r.get("bytes", 0) for r in results if r["status"] in ("downloaded", "skipped"))
total_mb /= 1_048_576

print(f"  Downloaded : {len(downloaded)} files")
print(f"  Skipped    : {len(skipped)} files")
print(f"  Errors     : {len(errors)} files")
print(f"  Total size : {total_mb:.1f} MB on disk")
print(f"  Finished   : {datetime.utcnow().isoformat()}Z")

if errors:
    print("  Error details:")
    for e in errors:
        print(f"    {e}")

StatementMeta(, a0441e3c-c954-4571-8460-6d374d5dac66, 15, Finished, Available, Finished, False)

  Downloaded : 0 files
  Skipped    : 12 files
  Errors     : 0 files
  Total size : 660.9 MB on disk
  Finished   : 2026-05-21T19:34:47.446447Z
